# Notebook 29: Einstein from Entropy (Paper III, §3–4)

This notebook verifies the entropy-to-Einstein chain:

1. **Strict concavity**: $d^2S/dC_1^2 = -(1/2)\sum 1/\lambda_m^2 < 0$
2. **$C_1$ affine in $R$**: $C_1 = (N{-}1)[1 - R\varepsilon^2/4 + O(\varepsilon^4)]$
3. **Onsager contraction**: factor $< 1$ for all $N \ge 7$
4. **Sobolev constant**: $C_S = 0.45$ on Bolza
5. **Non-crossing bound**: $R_{\max} = (N^2-2)/(4N^2) < 1/4$

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from math import sqrt, pi, log, sinh, cosh, tanh, exp, e

assertion_count = 0

## 1. Strict Concavity: $d^2S/dC_1^2 < 0$

The microcanonical entropy $S(C_1)$ is strictly concave:
$$\frac{d^2 S}{dC_1^2} = -\frac{1}{2} \sum_{m \ne m^*} \frac{1}{\lambda_m^2} < 0$$
where $\lambda_m = C_1 - m(N-m)/2$ are the Havelock eigenvalues.

In [ ]:
def casimir(m, N):
    return m * (N - m) / 2.0

def b_exact(N):
    return N * (N + 1) / 12 - log(2) + log(N) / (N - 1)

def havelock_eigenvalues(N, C1):
    """All Havelock eigenvalues at given C1."""
    return [C1 - casimir(m, N) for m in range(1, N)]

def d2S_dC1_sq(N, C1):
    """Second derivative of entropy wrt C1."""
    m_star = N // 2
    total = 0.0
    for m in range(1, N):
        if m == m_star:
            continue
        lam = C1 - casimir(m, N)
        if abs(lam) > 1e-12:
            total += 1.0 / lam**2
    return -0.5 * total

print('Strict concavity d^2S/dC1^2 = -(1/2) sum 1/lambda_m^2:\n')
print(f'{"N":>4s} {"C1":>8s} {"d2S/dC1^2":>14s} {"concave?":>10s}')
print('-' * 40)

for N in [7, 8, 11]:
    m_star = N // 2
    f_star = casimir(m_star, N)
    # Test at several C1 values above the critical mode Casimir
    for C1 in [f_star + 1, f_star + 5, f_star + 20]:
        d2S = d2S_dC1_sq(N, C1)
        assert d2S < 0, f'Concavity failed: N={N}, C1={C1}, d2S={d2S}'
        assertion_count += 1
        print(f'{N:4d} {C1:8.1f} {d2S:14.6f} {"YES":>10s}')
    print()

In [ ]:
# Plot S(C1) showing concavity for N=7
N = 7
m_star = N // 2
f_star = casimir(m_star, N)

def S_microcanonical(N, C1):
    """Microcanonical entropy: S = (1/2) sum log(lambda_m)."""
    m_star = N // 2
    total = 0.0
    for m in range(1, N):
        if m == m_star:
            continue
        lam = C1 - casimir(m, N)
        if lam > 1e-12:
            total += 0.5 * log(lam)
        else:
            return float('-inf')
    return total

print(f'S(C1) for N={N} (should be concave):\n')
print(f'{"C1":>8s} {"S(C1)":>12s} {"d2S":>12s}')
print('-' * 36)

C1_vals = np.linspace(f_star + 0.5, f_star + 30, 15)
S_prev = None
for C1 in C1_vals:
    S = S_microcanonical(N, C1)
    d2S = d2S_dC1_sq(N, C1)
    print(f'{C1:8.2f} {S:12.6f} {d2S:12.6f}')

print(f'\nAll d2S/dC1^2 values are negative: CONCAVE.')

## 2. $C_1$ Affine in $R$: Small-Ring Expansion

On $\mathbb{H}^2$ with curvature $K = -1/a^2$:
$$C_1 = (N{-}1)\left[1 - \frac{R\varepsilon^2}{4} + O(\varepsilon^4)\right]$$
where $R = K = -1/a^2$ and $\varepsilon = r_E/a$ is the ring radius ratio.

In [ ]:
from planetary_polygons.extensions.h2_stability import C1_h2_exact

print('C1 affine in R: verify small-epsilon expansion\n')
print('C1_exact(N, xi) = (N-1)(1+xi^2)/(1-xi)^2')
print('At small xi: C1 ~ (N-1)(1 + 2xi + ...) = (N-1)[1 + R*eps^2/(-4) + ...]')
print('where xi = eps^2 and R = -1/a^2 (H^2 curvature).\n')

for N in [7, 8]:
    print(f'N = {N}:')
    print(f'{"eps":>8s} {"xi=eps^2":>10s} {"C1_exact":>12s} {"(N-1)(1+2xi)":>14s} {"rel_err":>12s}')
    print('-' * 60)
    for eps in [0.01, 0.02, 0.05, 0.1, 0.2]:
        xi = eps**2
        C1_ex = C1_h2_exact(N, xi)
        # Leading expansion: C1 ~ (N-1)(1 + 2*xi + 3*xi^2 + ...)
        C1_approx = (N - 1) * (1 + 2 * xi)
        rel_err = abs(C1_ex - C1_approx) / C1_ex
        print(f'{eps:8.3f} {xi:10.6f} {C1_ex:12.6f} {C1_approx:14.6f} {rel_err:12.2e}')
    print()

# At small eps, the error should be O(xi^2) = O(eps^4)
eps_test = 0.01
xi_test = eps_test**2
for N in [7, 8]:
    C1_ex = C1_h2_exact(N, xi_test)
    C1_app = (N - 1) * (1 + 2 * xi_test)
    rel_err = abs(C1_ex - C1_app) / C1_ex
    # Error should scale as xi^2 ~ eps^4 ~ 1e-8
    assert rel_err < 1e-5, f'Expansion error too large: {rel_err}'
    assertion_count += 1

print('Small-ring expansion verified: error is O(eps^4) as expected.')

## 3. Onsager Contraction Factor

The Onsager contraction factor must satisfy $\gamma_{\text{contr}} < 1$
for the BO approximation to converge. Computed from the Sobolev bound
and the nonlinear potential.

In [ ]:
from planetary_polygons.extensions.onsager_selection import onsager_contraction

print('Onsager contraction factor for N=7..15:\n')
print(f'{"N":>4s} {"contraction":>14s} {"< 1?":>8s}')
print('-' * 30)

for N in range(7, 16):
    gamma = onsager_contraction(N)
    ok = gamma < 1.0
    assert ok, f'Contraction >= 1 at N={N}: {gamma}'
    assertion_count += 1
    print(f'{N:4d} {gamma:14.6f} {"YES" if ok else "NO":>8s}')

# Highlight N=7
gamma_7 = onsager_contraction(7)
print(f'\nN=7 contraction factor: {gamma_7:.4f}')
assert gamma_7 < 0.1, f'Expected gamma_7 < 0.1, got {gamma_7}'
assertion_count += 1
print('All contraction factors are strictly less than 1.')

## 4. Sobolev Constant on Bolza: $C_S \approx 0.45$

The heat-kernel Sobolev bound on the Bolza surface (genus 2, $K=-1$):
$$\|u\|_\infty^2 \le \frac{\lambda_1}{4\pi}\|u\|_2^2 + \frac{1}{e\lambda_1}\|\nabla u\|_2^2$$
with $\lambda_1 = 3.839$ (Buser 1992).

In [ ]:
from planetary_polygons.extensions.onsager_selection import (
    bolza_sobolev_bound, BOLZA_LAMBDA1, BOLZA_VOL
)

sup_bound, C_S, sigma_sq = bolza_sobolev_bound(var_R=2.0, grad_R_sq=2.0)

coeff1 = BOLZA_LAMBDA1 / (4 * pi)
coeff2 = 1.0 / (e * BOLZA_LAMBDA1)

print('Bolza surface Sobolev bound:\n')
print(f'lambda_1 = {BOLZA_LAMBDA1:.3f} (first nonzero Laplacian eigenvalue)')
print(f'Vol = {BOLZA_VOL:.6f} (area = 4*pi for genus 2, K=-1)')
print()
print(f'Coefficient 1: lambda_1/(4*pi) = {coeff1:.4f}')
print(f'Coefficient 2: 1/(e*lambda_1)  = {coeff2:.4f}')
print()
print(f'sup_bound  = sqrt({coeff1:.4f}*2 + {coeff2:.4f}*2) = {sup_bound:.4f}')
print(f'C_S        = {C_S:.4f}')

# Verify coefficients
assert abs(coeff1 - 0.306) < 0.01, f'Expected ~0.306, got {coeff1}'
assertion_count += 1
assert abs(coeff2 - 0.096) < 0.01, f'Expected ~0.096, got {coeff2}'
assertion_count += 1

# Verify sup_bound
expected_sup_sq = coeff1 * 2.0 + coeff2 * 2.0
expected_sup = sqrt(expected_sup_sq)
assert abs(sup_bound - expected_sup) < 1e-10
assertion_count += 1

# C_S should be approximately 0.45
assert abs(C_S - 0.45) < 0.05, f'Expected C_S ~ 0.45, got {C_S}'
assertion_count += 1
print(f'\nVerified: C_S = {C_S:.4f} (expected ~0.45).')

## 5. Non-Crossing Bound: $R_{\max} = (N^2 - 2)/(4N^2) < 1/4$

The maximum Ricci scalar for which polygons of different $N$ remain non-crossing
is bounded by $R_{\max} = (N^2 - 2)/(4N^2)$, strictly less than $1/4$ for all $N \ge 2$.

In [ ]:
from planetary_polygons.extensions.onsager_selection import non_crossing_bound

print('Non-crossing bound R_max = (N^2 - 2)/(4N^2):\n')
print(f'{"N":>4s} {"R_max":>12s} {"1/4 - R_max":>14s} {"< 1/4?":>8s}')
print('-' * 42)

for N in range(3, 21):
    R_max = non_crossing_bound(N)
    gap = 0.25 - R_max
    assert R_max < 0.25, f'Non-crossing bound failed at N={N}'
    assertion_count += 1
    print(f'{N:4d} {R_max:12.6f} {gap:14.6f} {"YES":>8s}')

# Verify asymptotic approach to 1/4
R_3 = non_crossing_bound(3)
R_20 = non_crossing_bound(20)
assert R_20 > R_3, 'R_max should increase with N'
assertion_count += 1
assert R_20 < 0.25, 'R_max should be strictly < 1/4'
assertion_count += 1

# Check exact formula
for N in [3, 5, 10, 20]:
    expected = (N**2 - 2) / (4 * N**2)
    actual = non_crossing_bound(N)
    assert abs(actual - expected) < 1e-15, f'Formula mismatch at N={N}'
    assertion_count += 1

print(f'\nR_max -> 1/4 as N -> infinity.')
print(f'R_max(3) = {R_3:.6f}, R_max(20) = {R_20:.6f}, limit = 0.250000')

In [ ]:
# Tabular display showing approach to 1/4
print('Approach to 1/4 limit:\n')
print(f'{"N":>4s} {"R_max":>12s} {"gap = 1/4 - R_max":>20s} {"gap ~ 1/(2N^2)":>16s}')
print('-' * 56)

for N in [3, 5, 10, 20, 50, 100]:
    R_max = non_crossing_bound(N)
    gap = 0.25 - R_max
    predicted_gap = 1.0 / (2 * N**2)
    print(f'{N:4d} {R_max:12.8f} {gap:20.8f} {predicted_gap:16.8f}')
    # Gap should be exactly 1/(2N^2)
    assert abs(gap - predicted_gap) < 1e-12, f'Gap formula mismatch at N={N}'
    assertion_count += 1

print(f'\nExact: 1/4 - R_max = 1/(2N^2) for all N.')

## Summary

1. **Strict concavity**: $d^2S/dC_1^2 < 0$ verified for $N = 7, 8, 11$ at multiple $C_1$ values.
   The entropy functional is strictly concave, ensuring a unique maximum.

2. **$C_1$ affine in $R$**: Small-ring expansion $C_1 = (N{-}1)(1 + 2\xi + \ldots)$ verified
   to $O(\varepsilon^4)$ accuracy for $N = 7, 8$.

3. **Onsager contraction**: All factors $< 1$ for $N = 7, \ldots, 15$.
   $N = 7$ contraction factor $\approx 0.04$.

4. **Sobolev constant**: $C_S \approx 0.45$ on Bolza, with coefficients $0.306$ and $0.096$.

5. **Non-crossing bound**: $R_{\max} = (N^2-2)/(4N^2)$ strictly less than $1/4$,
   with exact gap $1/(2N^2)$.

In [ ]:
print(f'\nAll {assertion_count} assertions passed.')